# Day 3 - Test splits

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split


In [ ]:
df = sns.load_dataset("titanic")

In [ ]:
# Prepare the test dataset
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["survived"], random_state=42
)
# sanity check the stratification held:
print(train_df["survived"].mean(), test_df["survived"].mean())

In [ ]:
df[df["pclass"] == 1].groupby("sex").value_counts()

In [ ]:
df.groupby(['pclass', 'sex']).size()

In [ ]:
df.groupby(['pclass','sex','sibsp','parch']).size()

In [ ]:
# Coarse grouping — 6 groups, all large (dozens–hundreds of members)
df['risk_coarse_leaky'] = df.groupby(['pclass','sex'])['survived'].transform('mean')

# Fine grouping — 74 groups, many with <=3 members
df['risk_fine_leaky'] = df.groupby(['pclass','sex','sibsp','parch'])['survived'].transform('mean')

In [ ]:
# Train logistic regression model on coarse grouping
from sklearn.linear_model import LogisticRegression

global_mean = train_df['survived'].mean()
coarse_means = train_df.groupby(['pclass', 'sex'])['survived'].mean()

# fit on train only, apply to test as a lookup — no leakage
train_df['risk_coarse'] = train_df.set_index(['pclass', 'sex']).index.map(coarse_means)
test_df['risk_coarse'] = test_df.set_index(['pclass', 'sex']).index.map(coarse_means).fillna(global_mean)

y_train = train_df['survived']
y_test = test_df['survived']

model_coarse = LogisticRegression().fit(train_df[['risk_coarse']], y_train)
acc_coarse = model_coarse.score(test_df[['risk_coarse']], y_test)
print(f"Coarse grouping test accuracy: {acc_coarse:.4f}")

In [ ]:
# Train logistic regression model on fine grouping (smoothed — 39 of 74 groups have <=3 members)

k = 10  # smoothing strength: how many "fake" global-average rows to blend in

fine_stats = train_df.groupby(['pclass', 'sex', 'sibsp', 'parch'])['survived'].agg(['mean', 'count'])
fine_stats['smoothed'] = (
    (fine_stats['count'] * fine_stats['mean'] + k * global_mean) / (fine_stats['count'] + k)
)
fine_lookup = fine_stats['smoothed']

# fit on train only, apply to test as a lookup — no leakage
train_df['risk_fine'] = train_df.set_index(['pclass', 'sex', 'sibsp', 'parch']).index.map(fine_lookup)
test_df['risk_fine'] = test_df.set_index(['pclass', 'sex', 'sibsp', 'parch']).index.map(fine_lookup).fillna(global_mean)

model_fine = LogisticRegression().fit(train_df[['risk_fine']], y_train)
acc_fine = model_fine.score(test_df[['risk_fine']], y_test)
print(f"Fine grouping test accuracy:   {acc_fine:.4f}")

# proba_fine = model_fine.predict_proba(test_df[['risk_fine']]).flatten()
# print(f"Fine grouping test log-loss: {proba_fine}")

In [ ]:
# Compare — honest (non-leaky) accuracy, coarse vs fine
print(f"Coarse grouping (honest):            {acc_coarse:.4f}")
print(f"Fine grouping   (honest, smoothed):  {acc_fine:.4f}")